# GEMS G02 — geological fault evidence: displacement, truncation and structural continuity

**Question (falsifiable).** Does label-free *structural* evidence — apparent offset of fabric across a line, termination of persistent ridges at a line, multiscale ridge continuity, lineament-network geometry and competing geological explanations — add equal-area fault proximity beyond G01 wavelet evidence and a simple magnetic gradient, **consistently across held-out geographic folds and discovery splits**?

| ID | candidate | physical idea | judged by |
|---|---|---|---|
| B0 | simple magnetic gradient (THDR) | contrast | — |
| B1 | best G01 magnetic-only map (nested choice) | multiscale magnetic energy | B1 vs B0 |
| B2 | best G01 fusion map (nested choice among G01-C/D/E/F) | joint mag + grav evidence | B2 vs B1, B0 |
| G02-A | displacement | fabric continues across the line but **offset** (step, not oblique slope) | standalone + G01 + G02-A vs G01 |
| G02-B | truncation | persistent ridges **end** along a common line | idem |
| G02-C | continuity | ridge position / strike stable across scales, straight, continuous | idem |
| G02-D | network | segment length, straightness, continuation (colinear / stepover), not forced connectivity | idem |
| G02-E | hypotheses | fault vs contact vs intrusive vs ambiguous support | idem |
| G02-F | G01 + selected G02 evidence (nested choice of base, family, weight) | complementarity | G02-F vs B2 |

**Rules of this notebook**
* Read-only on M9 and G01: cached layers are reused by name; no M9 section is run, no checkpoint touched, no G01 run modified (G01 maps are recomputed by G01's own code inside G02).
* Every G02 map is label-free. Labels are used only to evaluate and — nested, on *other* zones, excluding overlapping ones — to choose B1 / B2 / G02-F variants. No weight or threshold is fitted on a held-out zone.
* `FAST_RESEARCH` is exploratory (2 units per task, 2 wavelengths): it can never yield KEEP. Only `FULL_EVALUATION` gives recommendations.
* Apparent displacement ≠ geological slip. Hypothesis maps are scores, not probabilities.
* **No number exists until the cells run.**

Code: `gems_bench/m9/research/` · configuration: `configs/m9/g02_structural_geology.yaml` · guide: `docs/m9/G02_STRUCTURAL_GEOLOGY.md` · G01 review: `docs/m9/G01_REVIEW_AND_REUSE.md`.

## 0. Configuration, Drive and context

Same workflow as the G01 / M9 notebooks: mount Drive, install the repository at `CONFIG['ref']`, open the pipeline context (data stages validated / staged from Drive, never regenerated) and create the `m9` object **without running any M9 section**.

Prerequisite: M9 section 03 (`s03_spectral`) and, for G01-F, section 02 have run once, so `cache/m9/layers` (on Drive) holds the log-Gabor and cross-modal layers. If not, run the G01 notebook once with `build_missing = {'wavelets': True, 'crossmodal': True}`.

Set `G02_CONFIG['mode']`: start with `FAST_RESEARCH` (minutes per unit), then `FULL_EVALUATION` (all zones, all cached wavelengths, 500 bootstrap draws).

In [ ]:
CONFIG = {                                         # master-pipeline context (data stages are only validated / staged)
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'gems3-discovery-research',               # branch or commit SHA
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',
    'run_mode': 'PREPARE_DATA',
    'failure_policy': 'interactive',
    'download_missing': False,
    'run_advanced_models': False, 'train_baselines': [], 'run_full_inference': False, 'run_submission': False,
    'feature_stage': 'lazy',
    'use_gpu': False,                                # G02 is CPU only
    'seed': 2026,
}
DEM_CONFIG = {'enabled': False, 'mode': 'inventory_only', 'download': False}
PRIVATE_REPO = False

M9_CONFIG = {'profile': 'PILOT', 'overwrite_existing': False}   # only opens the M9 context; no section is run

G02_CONFIG = {
    'mode': 'FAST_RESEARCH',                         # FAST_RESEARCH (exploratory) | FULL_EVALUATION
    'overwrite': False,                              # True rewrites files inside the SAME fingerprinted run folder
    'g01': {'review_run': 'auto'},                   # persisted G01 run to review: auto (LATEST) | none | 'g01_<id>'
    'evaluation': {'tasks': ['geo', 'disc_official']},
    'baselines': {'m9_prediction_npy': None},        # optional: path to an EXISTING full-grid M9 prediction (.npy)
    'save': {'push_to_drive': True, 'keep_tmp': False},
}
SHOW_FIGURES = True

In [ ]:
import base64, os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = f"https://github.com/{CONFIG['repo']}.git"

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN') or ''
    hdr = base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {hdr}'}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args[:3])} failed:\n{r.stderr.strip()[-1500:]}")
    return r

if not os.path.exists(f'{CODE}/.git'):
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'remote', 'set-url', 'origin', REPO_URL)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
before = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if git('-C', CODE, 'rev-parse', '--verify', '-q', f'refs/remotes/origin/{ref}', check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', ref)
    git('-C', CODE, 'merge', '-q', '--ff-only', f'origin/{ref}')
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{CODE}[colab]'], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
print('code @', HEAD)
if 'gems_bench' in sys.modules and HEAD != before:
    raise RuntimeError('Code changed after gems_bench was imported: Runtime -> Restart session, then run from the top.')

In [ ]:
import json, pathlib
import numpy as np, pandas as pd
from IPython.display import Image, Markdown, display
from gems_bench.pipeline import Pipeline
from gems_bench.m9.research import G02Experiment

pipe = Pipeline.from_notebook(CONFIG, DEM_CONFIG)
pipe.step('bootstrap')
for s in ('raw_data', 'native_grid', 'spatial_folds', 'discovery_splits', 'feature_store'):
    pipe.step(s)                                     # validated on Drive and staged locally (never regenerated)

m9 = pipe.m9(M9_CONFIG)                              # context only: no M9 section, no model, no training
g02 = G02Experiment(m9, G02_CONFIG)                  # constructing it computes nothing
print('mode:', g02.cfg['mode'])

def show(names):
    if not SHOW_FIGURES:
        return
    for n in names:
        p = g02.store.root / 'figures' / f'{n}.png'
        display(Image(filename=str(p))) if p.exists() else print('not produced:', n)

def tab(name, n=40, query=None):
    t = g02.store.load_table(name, 'tables')
    if query and len(t):
        t = t.query(query)
    display(t.head(n)) if len(t) else print('empty / not produced:', name)
    return t

## 1. Inputs and the G01 review (what G01 actually established)

* **What.** Audit the cached layers (both fields per wavelength), choose the G02 scales, fingerprint every input and open the versioned run folder `…/research/G02_structural_geology/g02_<fp>/` with `config/ processed/ rasters/ lineaments/ tables/ figures/ reports/ manifests/`.
* **G01 review (read-only).** The newest persisted G01 run (or `g01.review_run`) is read and the ten review questions are answered **from its tables** — completed steps, informative wavelengths, gravity vs magnetic, strike gate, persistence, tolerance, vs edge, vs M9 cross-modal, fold consistency, redundancy with the gradient. Absent tables are reported as **ABSENT** and the component as **UNVERIFIED** — nothing is assumed.
* **Read.** `tables/g01_review_questions`, `tables/g01_decision_table`, `reports/G01_REVIEW.md`, `tables/g02_scales` (profile offsets, window lengths, search range and offset resolution per wavelength, in metres).

In [ ]:
info = g02.prepare()
print('run:', info['run_id'], '->', g02.store.root)
print('G02 wavelengths:', [w.label for w in info['wavelengths']])
for w in info['warnings']:
    print('WARNING', w)
rev = info['g01_review']
print('persisted G01 run found:', rev['available'])
display(rev['questions'])
display(rev['decision_table'])
tab('g02_scales')

In [ ]:
# Optional: review ANY G01 run folder (read-only; nothing is written into it)
from gems_bench.m9.research.g01_review import review_g01, g01_run_dirs
from gems_bench.m9.research.geology_io import run_root as g01_root
runs = g01_run_dirs(g01_root(m9, g02.g01cfg))
print('G01 runs:', [r.name for r in runs])
if runs:
    display(Markdown(review_g01(runs[-1])['markdown'][:6000]))

## 2. Held-out evaluation (tiled, resumable)

Per unit (geographic fold `geo`, discovery split `disc_official`):
1. Percentile references on the unit's **training geography** (`assert_disjoint` raises on any overlap with the evaluation zone).
2. Tiles (`tiling.tile_px`) with a halo computed from the scales → G01 maps (B0, G01-A..F, per-wavelength variants, M, G, J, P) and G02-A / B / C / E, written into memory-mapped arrays under `processed/tmp_<unit>/` (resumable per tile).
3. G02-D networks on the unit window from each `network.sources` evidence map → `lineaments/<unit>_<source>.geojson` (+ `.gpkg`), segment / node tables.
4. Equal-area counts per 20 km block for every candidate (seeded tie-break noise shared by all maps), complementarity, truth-length strata, label-free rank correlations, wavelength usage, lineament-geometry metrics, official-style DTI.
5. One streamed NPZ per unit `rasters/units/<unit>.npz` (float16 maps + domain + interior; labels never stored).

**Resume.** `processed/done_unit_*.json` per unit; an interrupted unit resumes at the next tile while its `tmp_` folder survives (local disk).

In [ ]:
units = g02.evaluate()
display(units)

## 3. Metrics, nested choices, contrasts, recommendations

* **Nested virtual candidates.** B1, B2, `F:<base>+<family>` (weight chosen) and G02-F (base, family and weight chosen) are picked for each held-out zone on the task's *other* zones, excluding zones whose evaluation footprint overlaps (`tables/unit_overlap`). Choices: `tables/nested_choices`.
* **Contrasts** (`tables/contrasts`, metric `decision.metric`; `tables/contrasts_precision` for false-positive reduction): paired spatial-block bootstrap at the primary budget / distance; SUPPORTED = Δ > 0, P(Δ > 0) ≥ 0.9 and ≥ 67 % of zones won; REFUTED = Δ < 0 and P(Δ > 0) ≤ 0.1.
  * `standalone`: family vs B0, random, B2 · `incremental`: base + family vs base (**the decision criterion**) · `field`: magnetic vs gravity · `band`: narrow vs broad · `network`: network on G01 evidence vs on the edge · `fusion`: G02-F vs B2.
* **Recommendations** (`tables/recommendations`): KEEP / INVESTIGATE / REJECT / UNVERIFIED from the incremental contrasts only (KEEP needs SUPPORTED on `disc_official` and no REFUTED elsewhere, in FULL_EVALUATION with ≥ 3 zones).
* **Complementarity** (`tables/complement_pooled`): truth missed by B0 (or a G01 base) and recovered by a candidate, relative to random at the same budget; precision of the candidate's new picks.
* **Geometry** (`tables/lineament_metrics`): segment-length precision, truth recall and orientation agreement at length budgets — separate from pixel ranking.

In [ ]:
res = g02.metrics()
display(res['recommendations'])
cols = ['family', 'kind', 'task', 'candidate', 'reference', 'delta', 'ci_lo', 'ci_hi', 'p_delta_positive',
        'zone_win_share', 'n_zones', 'verdict']
H = res['contrasts']
display(H[[c for c in cols if c in H]])
f0, d0 = g02.cfg['evaluation']['primary_area_frac'], g02.cfg['evaluation']['primary_distance_m']
P = res['pooled']
heads = ['B0', 'B1', 'B2', 'G02-A', 'G02-B', 'G02-C', 'G02-D', 'G02-E', 'G02-F', 'random']
display(P[(P.area_frac == f0) & (P.tol_m == d0) & P.candidate.isin(heads)]
        [['task', 'candidate', 'precision', 'recall', 'f1', 'enrichment', 'precision_vs_edge']])
display(res['complement'])
display(res['lineament'])

## 4. Figures (16:9, PNG + PDF + plotted data CSV in `figures/`)

Maps use one 50 km window centred on the held-out truth of the evaluated unit with the most truth (UTM 11N km; red = held-out truth, black = known faults: **evaluation figures**, labels never enter a map). Examples are seeded picks that include **strong and weak** evidence and strong gradients without G02 evidence.

1 raw fields + faults · 2 wavelet energy by scale · 3 orientation / coherence / discordance · 4 cross-fault profiles · 5 displacement examples · 6 truncation examples · 7 ridge continuity · 8 multiscale persistence · 9 candidate network · 10 G01 vs G02 maps · 11 TP / FP / missed case studies · 12 performance vs distance · 13 performance vs area · 14 per-fold improvements · 15 geological explanations + uncertainty.

In [ ]:
written = g02.figures()
show(['01_raw_fields_faults', '02_wavelet_energy', '03_orientation_coherence', '04_cross_fault_profiles',
      '05_displacement_examples', '06_truncation_examples', '07_ridge_continuity', '08_multiscale_persistence',
      '09_candidate_network', '10_g01_vs_g02', '11_case_studies', '12_performance_vs_distance',
      '13_performance_vs_area', '14_per_fold_improvements', '15_geological_explanations'])

## 5. Report and decision

`reports/G02_REPORT.md` is generated from the tables: recommendations, contrasts, pooled results and the nine decision questions (displacement vs gradient misses, truncation independence, persistence, false-positive reduction, wavelengths, magnetic vs gravity, complementarity with M9, regional consistency, integration). Copy the verdicts into `docs/m9/G02_STRUCTURAL_GEOLOGY.md` → *Results* by hand after reading figures 10, 11 and 14.

In [ ]:
rep = g02.report()
display(Markdown(rep.read_text()[:12000]))
g02.push()
print('local :', g02.store.root)
print('Drive :', m9.cfg.get('_drive_root'), '/', g02.cfg['output_subdir'], '/', g02.store.root.name)

## 6. Geological interpretation (write after reading the tables, not the maps)

* **G02-A.** Does displacement recover faults the gradient misses (`complement_pooled`, recovery vs random > 1) *and* add over G01 (incremental)? A standalone gain without an increment means it re-finds what G01 already finds.
* **G02-B.** Truncation should be weakly correlated with G01 (`rank_correlations`). High correlation + no increment = it is an energy-edge detector in disguise.
* **G02-C / D.** Do continuity / network raise *precision* (`contrasts_precision`)? A recall-only gain means they spread selections along lineaments rather than removing false positives.
* **G02-E.** Where is `ambiguous` dominant on truth (figure 11, 15)? That is the share of mapped faults the physics here cannot distinguish from other boundaries.
* **Consistency.** A gain carried by one zone (figure 14) is not a gain.
* **Integration.** Only KEEP from FULL_EVALUATION may be proposed for M7 / M9 post-processing — as a separate, pre-registered experiment. Nothing here touches a submission.

## Resume / reload / Drive synchronisation

* **Push.** Every unit, `metrics`, `figures` and `report` push the run folder to Drive (`m9.push`, only `research/G02_structural_geology/<run>`); the cell below pushes again explicitly.
* **Reload after a disconnect.** Run section 0, then either re-run any section (finished units / steps are reused from the fingerprinted folder; `stage_inputs` brings it back from Drive first) or attach and read results only.
* **Switch mode.** `FAST_RESEARCH` and `FULL_EVALUATION` have different fingerprints, hence different folders: they never overwrite each other.

In [ ]:
from gems_bench.m9.research.g02_io import list_runs, run_root
display(list_runs(run_root(m9, g02.cfg)))
g02_old = G02Experiment.attach(m9)                  # newest run; or attach(m9, run_id='g02_...')
old = g02_old.results()
display(old['recommendations'])
print(json.dumps(old['summary'], indent=1, default=str)[:3000])
g02_old.push()                                      # explicit Drive sync of that run folder